In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Task 1: Write your code here:
food_delivery_path = os.path.join(path, 'Q1_data.csv')
df_food_delivery = pd.read_csv(food_delivery_path)


In [ ]:
# Task 2: Write your code here:
df_food_delivery.head()


In [ ]:
# Task 3: Write your code here:
df_food_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_food_delivery.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food_delivery['Order_ID'].dropna()

In [ ]:
# Task 2: Write your code here:

df_food_delivery['Delivery_Time'].dropna() #Target column


#df_food_delivery['Weather'] = df_food_delivery['Weather'].fillna(df_food_delivery['Weather'].mode()[0])
#df_food_delivery['Traffic_Level'] = df_food_delivery['Traffic_Level'].fillna(df_food_delivery['Traffic_Level'].mode()[0])
#df_food_delivery['Time_of_Day'] = df_food_delivery['Time_of_Day'].fillna(df_food_delivery['Time_of_Day'].mode()[0])
#df_food_delivery['Courier_Experience_yrs'] = df_food_delivery['Courier_Experience_yrs'].fillna(df_food_delivery['Courier_Experience_yrs'].median()[0])

#Generates error..

#Those shouldnt be dropped but I dropped them to continue working,
df_food_delivery['Weather'].dropna()
df_food_delivery['Traffic_Level'].dropna()
df_food_delivery['Time_of_Day'].dropna()
df_food_delivery['Courier_Experience_yrs'].dropna()



In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_food_delivery)

In [ ]:
# Task 4: Write your code here:

categorical_cols = ['Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Weather']
for col in categorical_cols:
    le = LabelEncoder()
    df_food_delivery[col] = le.fit_transform(df_food_delivery[col].astype(str))
df_food_delivery.head()


In [ ]:
# Task 5: Write your code here
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_food_delivery[feature_cols]
y = df_food_delivery['Delivery_Time']

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_food_delivery[feature_cols]
y = df_food_delivery['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 2,3,4,5: Write your code here:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")


kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: